# TableRAG MVP: A Prototype for Table Question Answering

This notebook builds a small, step-by-step prototype inspired by [*TableRAG: Million-Token Table Understanding with Language Models*](https://arxiv.org/abs/2410.04739).

**Goal:** prepare a table for question answering without placing every row in an LLM prompt. TableRAG retrieves relevant column information and cell values, then gives that evidence to a solver.

## What this notebook builds

1. Load a table and keep the original data available.
2. Create a compact schema profile for its columns.
3. Build a frequency-ranked catalog of distinct column-value pairs, capped by a configurable budget.

Query expansion, retrieval, and answer generation are later components; they are not implemented yet.

> **Prototype note:** Titanic is a convenient small dataset for checking that the indexing steps work. It does not test TableRAG's performance or scalability on million-token tables.

In [16]:
import os
import pandas as pd

from pathlib import Path
from sklearn.datasets import fetch_openml


## 1. Load and Inspect the Titanic Data

We use the Titanic dataset identified by OpenML dataset ID `40945` as a small development table. The first run downloads it; subsequent runs can load the saved Parquet copy.

The DataFrame `df` is the complete source table. The schema profile and cell catalog created below are retrieval indexes; they do not replace or truncate the source data.

In [22]:
path = 'titanic.parquet'

path = Path("titanic.parquet")

if path.exists():
    df = pd.read_parquet(path)
else:
    data = fetch_openml(data_id=40945, as_frame=True)
    df = data.frame
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path)

display(df.shape)
display(df.head())

(1309, 14)

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## 2. Build the Schema Profile

The schema profile records each column's name, an automatically inferred type, and compact examples. It follows the paper's approach: numeric and datetime columns are summarized by their minimum and maximum, while categorical columns show up to three of their most frequent values.

Missing values are excluded when calculating examples. The type inference is a first pass based on pandas dtypes, so it may not capture meaning on its own: for example, `pclass` is stored numerically but represents ticket classes. We can add domain-specific type overrides if needed. Human-written column descriptions are optional and are separate from this profile.

In [27]:
def build_schema_metadata(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for column in df.columns:
        series = df[column]
        values = series.dropna() # Remove any None-values

        if pd.api.types.is_datetime64_any_dtype(series):
            column_type = "datetime"
            examples = (
                {"min": values.min(), "max": values.max()}
                if not values.empty else {}
            )
        elif pd.api.types.is_numeric_dtype(series):
            column_type = "numeric"
            examples = (
                {"min": values.min(), "max": values.max()}
                if not values.empty else {}
            )
        else:
            column_type = "categorical"
            examples = [
                {"value": value, "count": int(count)}
                for value, count in values.value_counts().head(3).items()
            ]

        rows.append({
            "column": column,
            "type": column_type,
            "examples": examples,
        })

    return pd.DataFrame(rows).set_index("column")

schema_metadata = build_schema_metadata(df)
display(schema_metadata)

,type,examples
column,,
pclass,numeric,"{'min': 1.0, 'max': 3.0}"
survived,categorical,"[{'value': '0', 'count': 809}, {'value': '1', ..."
name,categorical,"[{'value': 'Connolly, Miss. Kate', 'count': 2}..."
sex,categorical,"[{'value': 'male', 'count': 843}, {'value': 'f..."
age,numeric,"{'min': 0.1667, 'max': 80.0}"
sibsp,numeric,"{'min': 0.0, 'max': 8.0}"
parch,numeric,"{'min': 0.0, 'max': 9.0}"
ticket,categorical,"[{'value': 'CA. 2343', 'count': 11}, {'value':..."
fare,numeric,"{'min': 0.0, 'max': 512.3292}"


## 3. Build the Frequency-Ranked Cell Catalog

For every column, count each distinct non-missing value and store it as a `(column, value, frequency)` entry. Entries are ranked by frequency, and the catalog is limited to the first `B` pairs, following the paper's frequency-aware cell indexing approach.

The same value in two different columns is represented by two separate pairs. The budget limits the size of this retrieval catalog; the full DataFrame remains available for later calculations. In this Titanic run, the catalog contains 3,343 pairs, below the configured budget of 10,000, so no pairs are removed by the cap. You can temporarily lower `B` to verify truncation.

In [37]:
B = 10000 # Budget outlined in the paper

pairs = []

for col in df.columns:
    counts = df[col].value_counts(dropna=True) # returns a series containing the frequency of each unique value in col

    for val, freq in counts.items():
        if freq > 0:
            pairs.append({
                "column": col,
                "value": val,
                "frequency": int(freq)
            })

cell_catalog = (
    pd.DataFrame(pairs)
    .sort_values("frequency", ascending=False, kind="stable")
    .head(B)
    .reset_index(drop=True)
    .set_index("column")
)

display(cell_catalog.head(10))
print(f"Catalog entries: {len(cell_catalog)} (budget: {B})")


,value,frequency
column,,
parch,0.0,1002
embarked,S,914
sibsp,0.0,891
sex,male,843
survived,0,809
pclass,3.0,709
survived,1,500
sex,female,466
pclass,1.0,323


Catalog entries: 3343 (budget: 10000)
